# How LLMs See Text: Tokens, Context Windows, and Your First API Call

You've used LLMs. But do you know how they actually receive text? This notebook covers the mechanics — tokenization, context limits, and making your first direct API call — so the rest of the series makes sense.

**3 exercises.** Each is 2–5 lines of code. The goal isn't to write a lot — it's to understand exactly what those lines are doing.

### What you'll build

| Concept | What it means | Why it matters |
|---------|--------------|----------------|
| Tokens | Subword units the model actually processes | Every cost and limit is measured in tokens |
| Context window | The model's total working memory | Determines how much you can send at once |
| API call | The message format you send the model | The interface every LLM exposes |
| Temperature | How random the model's output is | Tune for fact vs. creativity |
| Token budget | Arithmetic for fitting context | Core skill for building RAG systems |

### Prerequisites
- Notebook 01 complete
```bash
pip install google-generativeai tiktoken numpy matplotlib
```

In [ ]:
import os
import json
import tiktoken
import google.generativeai as genai
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

# httpx SSL patch for corporate VPN environments
import httpx
_orig_client = httpx.Client.__init__
def _patched_client(self, *args, **kwargs):
    kwargs["verify"] = False
    _orig_client(self, *args, **kwargs)
httpx.Client.__init__ = _patched_client
_orig_async = httpx.AsyncClient.__init__
def _patched_async(self, *args, **kwargs):
    kwargs["verify"] = False
    _orig_async(self, *args, **kwargs)
httpx.AsyncClient.__init__ = _patched_async

plt.rcParams['figure.figsize'] = (13, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print("Setup complete.")

In [ ]:
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")
if not GEMINI_API_KEY:
    raise ValueError(
        "Set GEMINI_API_KEY environment variable before running this notebook.\n"
        "In terminal: export GEMINI_API_KEY='your-key-here'\n"
        "Then restart the Jupyter kernel."
    )
genai.configure(api_key=GEMINI_API_KEY)
print("Gemini configured.")

---
## Part 1: Tokens Are Not Words

LLMs don't process characters, and they don't process words. They process **tokens** — subword units that a tokenizer has learned from a large corpus.

A few facts that will save you confusion later:

- **Tokens ≈ 0.75 words on average** — so "4096 tokens" ≈ 3000 words ≈ 6 pages of text
- Common words are single tokens: `"the"` → 1 token
- Rare words split into pieces: `"unhappiness"` → `["un", "happiness"]` or even `["un", "hap", "piness"]`
- Code tokens are dense: `"print("` might be 2 tokens; a UUID is ~12 tokens
- The tokenizer is **fixed** — it's baked into the model and doesn't change

`tiktoken` is OpenAI's tokenizer library. The `cl100k_base` encoding is used by GPT-4 and many similar models. We'll use it throughout this notebook to count tokens.

In [ ]:
enc = tiktoken.get_encoding("cl100k_base")

text = "The learning rate controls gradient descent step size."
tokens = enc.encode(text)
decoded = [enc.decode([t]) for t in tokens]

print(f"Text:            {repr(text)}")
print(f"Token IDs:       {tokens}")
print(f"Decoded tokens:  {decoded}")
print()
print(f"Characters: {len(text)}")
print(f"Words:      {len(text.split())}")
print(f"Tokens:     {len(tokens)}")
print(f"Chars/token ratio: {len(text)/len(tokens):.2f}")

In [ ]:
def draw_token_boxes(text: str, title: str = "Tokenization"):
    """Visualize each token as a colored box with its text inside."""
    enc_local = tiktoken.get_encoding("cl100k_base")
    token_ids = enc_local.encode(text)
    decoded_tokens = [enc_local.decode([t]) for t in token_ids]

    COLORS = ['#4C9BE8', '#5DBE7C', '#E8A040', '#E8704C', '#9B59B6']

    fig, ax = plt.subplots(figsize=(13, 2.8))
    ax.set_xlim(0, 13)
    ax.set_ylim(0, 1)
    ax.axis('off')

    x = 0.1
    y_top = 0.85
    box_h = 0.55
    max_x = 12.8
    row = 0

    for i, (tok, tok_id) in enumerate(zip(decoded_tokens, token_ids)):
        color = COLORS[i % len(COLORS)]
        label = repr(tok)[1:-1]   # strip outer quotes, keep escape sequences readable
        char_w = max(len(label) * 0.085, 0.25)
        box_w = char_w + 0.18

        if x + box_w > max_x:
            x = 0.1
            row += 1

        y = y_top - row * 0.65
        rect = mpatches.FancyBboxPatch(
            (x, y - box_h / 2), box_w, box_h,
            boxstyle="round,pad=0.03", lw=1.5,
            edgecolor=color, facecolor=color + "30"
        )
        ax.add_patch(rect)
        ax.text(x + box_w / 2, y + 0.08, label,
                ha='center', va='center', fontsize=8, fontweight='bold', color='#333')
        ax.text(x + box_w / 2, y - 0.18, str(tok_id),
                ha='center', va='center', fontsize=6, color='#888')
        x += box_w + 0.08

    ax.set_title(
        f"{title}   [{len(token_ids)} tokens, {len(text)} chars, {len(text.split())} words]",
        fontsize=10, fontweight='bold', pad=6
    )
    plt.tight_layout()
    plt.show()

# Show two examples
draw_token_boxes("The learning rate controls gradient descent step size.", "Simple ML sentence")
draw_token_boxes("unhappiness tokenization GPT-4 https://api.openai.com/v1/chat/completions", "Rare words + URL")

---
### ✏️ Exercise 1: Token counting

Count tokens for three different text types and compare the character-to-token ratio. Which type of text is most token-efficient?

**Your task:** Implement `count_tokens(text)` — it should return the number of tokens in the string.

**Hint:** `enc.encode(text)` returns a list of token IDs.

In [ ]:
def count_tokens(text: str) -> int:
    """Return the number of tokens in text using the cl100k_base encoder."""
    # ✏️ YOUR TURN: one line using enc.encode()
    raise NotImplementedError("Implement count_tokens — use enc.encode(text)")

# ── Three test texts ──────────────────────────────────────────────────────────
english_prose = (
    "Gradient descent is an optimization algorithm that iteratively adjusts "
    "parameters to minimize a loss function by moving in the direction of the "
    "negative gradient."
)
python_code = (
    "def gradient_descent(X, y, lr=0.01, epochs=1000):\n"
    "    weights = np.zeros(X.shape[1])\n"
    "    for _ in range(epochs):\n"
    "        grad = X.T @ (X @ weights - y) / len(y)\n"
    "        weights -= lr * grad\n"
    "    return weights"
)
url_string = (
    "https://storage.googleapis.com/hankstank-models/mlb_ensemble_v8/"
    "artifacts/feature_importance_89feat_20260101.json"
)

# ── Tests ─────────────────────────────────────────────────────────────────────
n_prose = count_tokens(english_prose)
n_code  = count_tokens(python_code)
n_url   = count_tokens(url_string)

assert isinstance(n_prose, int), "count_tokens must return an int"
assert n_prose > 0 and n_code > 0 and n_url > 0, "Token counts should be positive"

print("Token-efficiency comparison")
print(f"{'Type':<20} {'Chars':>6} {'Tokens':>7} {'Chars/Token':>12}")
print("-" * 48)
for label, text, n in [
    ("English prose", english_prose, n_prose),
    ("Python code",   python_code,   n_code),
    ("URL/path",      url_string,    n_url),
]:
    ratio = len(text) / n
    print(f"{label:<20} {len(text):>6} {n:>7} {ratio:>11.2f}x")

print()
print("English prose is most token-efficient (~4.5–5 chars/token).")
print("Code and URLs are less efficient — unusual subwords = more splits.")
print("✅ Passed!")

<details>
<summary>💡 Solution (click to expand)</summary>

```python
def count_tokens(text: str) -> int:
    return len(enc.encode(text))
```

That's it. `enc.encode()` returns a list of integer token IDs — `len()` gives the count.

**Why this matters:** Every LLM API call is billed per token, not per character or word. The token count also determines whether your prompt fits in the context window.

</details>

---
## Part 2: Context Windows

The context window is the model's **total working memory** — everything you send (system prompt + conversation history + retrieved documents) must fit within it.

| Model | Context window |
|-------|---------------|
| Gemini 2.0 Flash | 1,048,576 tokens (~750K words) |
| GPT-4o | 128,000 tokens (~96K words) |
| Claude Sonnet 3.7 | 200,000 tokens (~150K words) |
| Local models (Mistral 7B) | 4,096–8,192 tokens (~3K–6K words) |

**The constraint that matters for RAG:** if you're deploying a system where the server runs a 4k–8k token local model (like in the `hank_agent` project), you have very little room. You might only fit 3–5 retrieved chunks alongside the system prompt and conversation.

### Practical calculation

A 4k context has:
- ~500 tokens for system prompt
- ~200 tokens for the user message
- Leaves 3,300 tokens for retrieved chunks and the model's answer
- At 200 tokens per chunk → only **16 chunks maximum**, and you want to leave ~500 tokens for the answer → **13 chunks**

In [ ]:
# Practical token budget arithmetic
CONTEXT_LIMIT = 8192  # tokens (small local model)

system_prompt = """You are a helpful assistant specializing in baseball analytics.
Answer questions using only the provided context. If the context doesn't contain
the answer, say so clearly. Format numbers to 3 decimal places.""".strip()

user_message = "What was the highest ERA in the 2025 MLB season among starters with 20+ starts?"

# Simulate 3 retrieved document chunks
chunks = [
    "Jacob deGrom posted a 1.92 ERA in 22 starts during the 2025 season, leading all qualified starters...",
    "Qualifying criteria for ERA title requires 162 innings pitched over the course of the season...",
    "The 2025 season saw a league-wide ERA of 4.21, up from 4.05 the prior year, driven by offense surge...",
]

system_tokens  = len(enc.encode(system_prompt))
message_tokens = len(enc.encode(user_message))
chunk_tokens   = [len(enc.encode(c)) for c in chunks]
total_chunk_t  = sum(chunk_tokens)
reserve_answer = 300

print(f"Context limit:    {CONTEXT_LIMIT:>6} tokens")
print(f"System prompt:    {system_tokens:>6} tokens")
print(f"User message:     {message_tokens:>6} tokens")
print(f"3 RAG chunks:     {total_chunk_t:>6} tokens  ({chunk_tokens})")
print(f"Answer reserve:   {reserve_answer:>6} tokens")
used = system_tokens + message_tokens + total_chunk_t + reserve_answer
print(f"─────────────────────────────")
print(f"Total used:       {used:>6} tokens")
print(f"Remaining:        {CONTEXT_LIMIT - used:>6} tokens")
print(f"Utilization:      {used/CONTEXT_LIMIT:>6.1%}")

# ─── Visualization ────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 2.5))
ax.set_xlim(0, CONTEXT_LIMIT)
ax.set_ylim(0, 1)
ax.axis('off')

segments = [
    ("System prompt",   system_tokens,  '#4C9BE8'),
    ("User message",    message_tokens, '#5DBE7C'),
    ("RAG chunk 1",     chunk_tokens[0], '#E8A040'),
    ("RAG chunk 2",     chunk_tokens[1], '#E8704C'),
    ("RAG chunk 3",     chunk_tokens[2], '#9B59B6'),
    ("Answer reserve",  reserve_answer, '#aaaaaa'),
    ("Free",            CONTEXT_LIMIT - used, '#e8e8e8'),
]

x = 0
for label, width, color in segments:
    rect = mpatches.FancyBboxPatch(
        (x, 0.2), width, 0.6,
        boxstyle="square,pad=0", lw=0.5,
        edgecolor='white', facecolor=color
    )
    ax.add_patch(rect)
    if width > 150:
        ax.text(x + width / 2, 0.5, f"{label}\n{width}t",
                ha='center', va='center', fontsize=7, fontweight='bold',
                color='white' if color not in ('#e8e8e8', '#aaaaaa') else '#555')
    x += width

ax.text(0, 0.95, f"8192-token context window", fontsize=9, color='#555')
ax.text(CONTEXT_LIMIT, 0.95, f"{CONTEXT_LIMIT - used} tokens free",
        fontsize=9, color='#555', ha='right')
ax.set_title("Context Window Budget", fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Part 3: Your First API Call

An LLM API call is just sending a message and getting text back. Under the hood it's an HTTP request with a JSON body, but the SDK wraps all that. The interface is:

1. Create a model object with the model name
2. Call `generate_content(prompt)` with your message
3. Read the response text from `.text`

That's the whole interface. The response object also carries metadata — how many tokens were used, which is how you get billed.

In [ ]:
model = genai.GenerativeModel("gemini-2.0-flash")

response = model.generate_content("What is gradient descent in one sentence?")
print("Response text:")
print(response.text)
print()
print("Response object attributes:")
print(f"  .text type:         {type(response.text)}")
if hasattr(response, 'usage_metadata') and response.usage_metadata:
    um = response.usage_metadata
    print(f"  prompt_token_count: {um.prompt_token_count}")
    print(f"  candidates_token_count: {um.candidates_token_count}")
    print(f"  total_token_count:  {um.total_token_count}")

---
### ✏️ Exercise 2: Make your own API call

Ask the model to explain what a token is. Print both the response text and the total number of tokens used.

**Your task:** Fill in the `ask_about_tokens()` function — it should make a Gemini API call and return `(response_text, total_tokens_used)`. If `usage_metadata` is unavailable, return `(response_text, -1)`.

**Hint:** Use `model.generate_content(prompt)` and check `response.usage_metadata.total_token_count`.

In [ ]:
def ask_about_tokens() -> tuple:
    """
    Ask the model to explain what a token is in LLMs.
    Returns: (response_text: str, total_tokens_used: int)
    If usage_metadata is unavailable, return (response_text, -1).
    """
    prompt = "In 2–3 sentences, explain what a 'token' is in the context of large language models."

    # ✏️ YOUR TURN:
    # 1. Call model.generate_content(prompt)
    # 2. Extract response.text
    # 3. Try to get response.usage_metadata.total_token_count (may be None)
    # 4. Return (text, total_token_count)
    raise NotImplementedError("Implement ask_about_tokens — make the Gemini API call")

# ── Test ──────────────────────────────────────────────────────────────────────
text, total_tokens = ask_about_tokens()

assert isinstance(text, str), "Response text must be a string"
assert len(text) > 20, "Response should be a non-empty explanation"

print("Model's explanation of tokens:")
print(text)
print()
print(f"Total tokens used: {total_tokens}")
print("✅ Passed!")

<details>
<summary>💡 Solution (click to expand)</summary>

```python
def ask_about_tokens() -> tuple:
    prompt = "In 2–3 sentences, explain what a 'token' is in the context of large language models."
    response = model.generate_content(prompt)
    text = response.text
    try:
        total_tokens = response.usage_metadata.total_token_count
    except (AttributeError, TypeError):
        total_tokens = -1
    return (text, total_tokens)
```

**The pattern you'll use everywhere:** `model.generate_content()` → `.text` for the result, `.usage_metadata` for billing/budget tracking. Every production system that uses an LLM API logs token counts to track cost.

</details>

---
## Part 4: Temperature and Sampling

After a forward pass, the model produces a **probability distribution** over its entire vocabulary (~100k tokens). Temperature controls how that distribution is used to pick the next token:

- **Temperature = 0:** always pick the highest-probability token (greedy / deterministic)
- **Temperature = 0.7:** sample from the distribution, but still prefer higher-probability tokens
- **Temperature = 1.5:** flatten the distribution, making rare tokens more likely → creative but sometimes incoherent
- **Temperature > 2.0:** the distribution becomes nearly uniform → very unpredictable

**Rule of thumb:**
- Factual tasks (Q&A, summarization, extraction): `temperature = 0.0–0.3`
- Balanced tasks (chat, instruction-following): `temperature = 0.7`
- Creative tasks (brainstorming, poetry): `temperature = 1.0–1.5`

In [ ]:
# Visualize how temperature reshapes the probability distribution
# (mock distribution — illustrates the concept without an API call)

np.random.seed(42)
token_labels = ["blue", "clear", "dark", "bright", "cloudy", "grey", "red", "vast"]
raw_logits = np.array([3.2, 2.1, 1.5, 1.2, 0.8, 0.4, 0.1, -0.5])  # mock logits

def softmax_with_temp(logits, temp):
    if temp == 0:
        # Argmax: one-hot
        probs = np.zeros_like(logits)
        probs[np.argmax(logits)] = 1.0
        return probs
    scaled = logits / temp
    scaled -= scaled.max()   # numerical stability
    exp_s = np.exp(scaled)
    return exp_s / exp_s.sum()

temps = [0.0, 0.7, 1.5]
temp_colors = ['#4C9BE8', '#E8A040', '#E8704C']
temp_labels = ['temp=0 (greedy)', 'temp=0.7 (balanced)', 'temp=1.5 (creative)']

fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
for ax, t, color, label in zip(axes, temps, temp_colors, temp_labels):
    probs = softmax_with_temp(raw_logits, t)
    bars = ax.bar(token_labels, probs, color=color, alpha=0.85, edgecolor='white', lw=0.8)
    ax.set_title(label, fontsize=9.5, fontweight='bold')
    ax.set_ylabel("Probability" if ax == axes[0] else "")
    ax.set_ylim(0, 1.05)
    ax.tick_params(axis='x', rotation=30, labelsize=8)
    # Annotate top bar
    top_idx = np.argmax(probs)
    ax.annotate(
        f'p={probs[top_idx]:.2f}',
        (top_idx, probs[top_idx]),
        xytext=(0, 6), textcoords='offset points',
        ha='center', fontsize=8, color=color, fontweight='bold'
    )
    # Show "The sky is ___" prompt context
    sampled = np.random.choice(token_labels, p=probs)
    ax.set_xlabel(f'→ sampled: "{sampled}"', fontsize=8.5, style='italic', color='#555')

fig.suptitle(
    'Prompt: "Complete this sentence with one word: The sky is ___"\n'
    'How temperature reshapes the next-token probability distribution',
    fontsize=10, fontweight='bold'
)
plt.tight_layout()
plt.show()

In [ ]:
# Run the same prompt at three temperatures — see how outputs vary
prompt = "Complete this sentence with exactly one word: The sky is"

print(f"Prompt: '{prompt}'\n")
print(f"{'Temperature':<14} {'Run 1':<20} {'Run 2':<20} {'Run 3':<20}")
print("─" * 74)

for temp in [0.0, 0.7, 1.5]:
    config = genai.GenerationConfig(temperature=temp, max_output_tokens=8)
    results = []
    for _ in range(3):
        r = model.generate_content(prompt, generation_config=config)
        results.append(r.text.strip().split()[0] if r.text.strip() else "(empty)")
    print(f"temp={temp:<9} {results[0]:<20} {results[1]:<20} {results[2]:<20}")

print()
print("At temp=0: same word every run (deterministic).")
print("At temp=1.5: different words each run — more creative, less predictable.")

---
### ✏️ Exercise 3: Temperature experiment

Run a creative prompt at temperature 0.0 and 2.0, three times each. What do you notice about consistency vs. variety?

**Your task:** Complete the `temperature_experiment()` function — make 3 `generate_content` calls per temperature and collect the results.

In [ ]:
def temperature_experiment(prompt: str, temperatures: list, n_runs: int = 3) -> dict:
    """
    Run the same prompt at each temperature n_runs times.
    Returns: dict mapping temperature (float) -> list of response strings (len == n_runs)
    """
    results = {}
    for temp in temperatures:
        config = genai.GenerationConfig(temperature=temp, max_output_tokens=20)
        runs = []
        for _ in range(n_runs):
            # ✏️ YOUR TURN:
            # 1. Call model.generate_content(prompt, generation_config=config)
            # 2. Append the .text.strip() to runs
            raise NotImplementedError("Make the generate_content call and append result to runs")
        results[temp] = runs
    return results

# ── Test ──────────────────────────────────────────────────────────────────────
creative_prompt = "Give me a one-word adjective to describe the color of the ocean."
experiment = temperature_experiment(creative_prompt, temperatures=[0.0, 2.0], n_runs=3)

print(f"Prompt: '{creative_prompt}'\n")
for temp, runs in experiment.items():
    print(f"Temperature {temp}:")
    for i, r in enumerate(runs):
        print(f"  Run {i+1}: {r}")

# Low temp should be consistent (all same), high temp should vary
low_runs  = experiment[0.0]
high_runs = experiment[2.0]

assert len(low_runs) == 3 and len(high_runs) == 3, "Should have 3 runs each"
assert all(isinstance(r, str) and len(r) > 0 for r in low_runs + high_runs), "All results should be non-empty strings"

# At temp=0, results should all be the same (or very close)
low_consistent = len(set(low_runs)) == 1
print()
print(f"Low temp (0.0) all identical: {low_consistent}")
print(f"High temp (2.0) unique responses: {len(set(high_runs))}/3")
print("✅ Passed!")

<details>
<summary>💡 Solution (click to expand)</summary>

```python
def temperature_experiment(prompt: str, temperatures: list, n_runs: int = 3) -> dict:
    results = {}
    for temp in temperatures:
        config = genai.GenerationConfig(temperature=temp, max_output_tokens=20)
        runs = []
        for _ in range(n_runs):
            r = model.generate_content(prompt, generation_config=config)
            runs.append(r.text.strip())
        results[temp] = runs
    return results
```

**Note:** Gemini at `temperature=0` is not perfectly deterministic (unlike some other models), so the low-temp runs might vary slightly. The principle holds — low temperature produces much more consistent output than high temperature.

</details>

---
## Part 5: Token Budget for RAG

Every RAG system is a token budget problem. You have N tokens total. Your system prompt costs some. Each retrieved chunk costs some. You need to leave enough for the answer.

The failure mode: you retrieve 20 chunks, naively concatenate them, and the total exceeds the context limit. The API either truncates your prompt or returns an error. The result is a hallucinated or incomplete answer.

The fix: **count tokens before sending**, and drop chunks that don't fit.

In [ ]:
class TokenBudget:
    """Track token usage against a fixed budget."""

    def __init__(self, total: int):
        self.total = total
        self.used = 0
        self._items = []

    def add(self, text: str, label: str) -> bool:
        """
        Try to add text to the budget.
        Returns True if it fits, False if it would exceed the budget.
        """
        tokens = len(enc.encode(text))
        if self.used + tokens > self.total:
            print(f"  ✗ '{label}' ({tokens} tokens) doesn't fit — {self.total - self.used} remaining")
            return False
        self.used += tokens
        self._items.append((label, tokens))
        print(f"  ✓ '{label}' ({tokens} tokens) — {self.total - self.used} remaining")
        return True

    def summary(self):
        print(f"\nTotal: {self.total} | Used: {self.used} | Free: {self.total - self.used} ({(self.used/self.total)*100:.1f}% full)")


# ── Demo: simulate building a RAG prompt for a local 4096-token model ─────────
print("Simulating a 4096-token budget (local model scenario):\n")
budget = TokenBudget(total=4096)

system_p = (
    "You are a baseball analytics assistant. Answer questions using only the "
    "provided context. Cite specific statistics when available. If you cannot "
    "answer from the context, say so directly."
)
budget.add(system_p, "system_prompt")

user_q = "Which pitchers had the best ERA in the 2025 MLB season?"
budget.add(user_q, "user_question")

# Simulate retrieved RAG chunks of varying sizes
rag_chunks = [
    ("Chunk 1 (ERA leaders)", "Jacob deGrom led all starters with a 1.92 ERA in 22 starts during the 2025 season. " * 4),
    ("Chunk 2 (ERA context)",  "The league-wide ERA in 2025 was 4.21. The ERA title requires 162 innings pitched. " * 5),
    ("Chunk 3 (relievers)",    "Among relievers, Edwin Diaz posted a 1.45 ERA while striking out 14.2 batters per 9. " * 4),
    ("Chunk 4 (historical)",   "Historically, sub-2.00 ERAs are extremely rare — only 8 pitchers achieved this since 2000. " * 6),
    ("Chunk 5 (overflow)",     "Additional context about pitching statistics and advanced metrics from the 2025 season. " * 8),
]

print()
for label, text in rag_chunks:
    budget.add(text, label)

budget.add("(reserve for model answer)", " " * 1)  # placeholder to show remaining
budget.summary()

---
### ✏️ Exercise 4: Max chunks that fit

Given a token budget, a system prompt, and a list of chunks, figure out the maximum number of chunks (from the list, in order) that you can fit.

**Your task:** Implement `max_chunks_that_fit()` — return the count of chunks that fit within `budget - len(system_prompt tokens)`. Leave 300 tokens reserved for the model's answer.

**Hint:** Use `count_tokens()` from Exercise 1 (or `len(enc.encode(text))`). Add chunks greedily in order until the next one would exceed the budget.

In [ ]:
def max_chunks_that_fit(total_budget: int, system_prompt: str, chunks: list,
                        answer_reserve: int = 300) -> int:
    """
    Return the number of chunks (from the front of the list) that fit within the budget.
    Budget breakdown: total_budget = system_prompt tokens + chunk tokens + answer_reserve
    Chunks are added greedily in order.
    """
    # ✏️ YOUR TURN:
    # 1. Count tokens in system_prompt, subtract from total_budget, subtract answer_reserve
    # 2. Greedily add chunks until the next one would not fit
    # 3. Return the count of chunks that did fit
    raise NotImplementedError("Implement max_chunks_that_fit")

# ── Test ──────────────────────────────────────────────────────────────────────
test_system = "You are a helpful assistant. Answer using only the provided context."
test_chunks = [
    "Chunk A: " + "This is a short chunk about gradient descent. " * 3,            # ~30 tokens
    "Chunk B: " + "This is a medium-length chunk about learning rates. " * 8,      # ~70 tokens
    "Chunk C: " + "This is a longer chunk about neural network architectures. " * 15, # ~130 tokens
    "Chunk D: " + "This is a very long chunk with lots of details about SGD. " * 20,  # ~165 tokens
    "Chunk E: " + "This final chunk would overflow a small budget easily. " * 30,     # ~250 tokens
]

result_4096 = max_chunks_that_fit(4096, test_system, test_chunks)
result_small = max_chunks_that_fit(300, test_system, test_chunks)

assert isinstance(result_4096, int), "Return type must be int"
assert result_4096 >= 1, "At least one chunk should fit in 4096 tokens"
assert result_small < result_4096, "Smaller budget should fit fewer chunks"
assert result_small >= 0, "Result must be non-negative"

print(f"4096-token budget: fits {result_4096} of {len(test_chunks)} chunks")
print(f" 300-token budget: fits {result_small} of {len(test_chunks)} chunks")

# Show the breakdown for the 4096-token case
budget_check = TokenBudget(total=4096)
budget_check.add(test_system, "system_prompt")
for i in range(result_4096):
    budget_check.add(test_chunks[i], f"Chunk {i+1}")
budget_check.summary()
print("✅ Passed!")

<details>
<summary>💡 Solution (click to expand)</summary>

```python
def max_chunks_that_fit(total_budget: int, system_prompt: str, chunks: list,
                        answer_reserve: int = 300) -> int:
    system_tokens = len(enc.encode(system_prompt))
    remaining = total_budget - system_tokens - answer_reserve
    count = 0
    for chunk in chunks:
        chunk_tokens = len(enc.encode(chunk))
        if remaining - chunk_tokens < 0:
            break
        remaining -= chunk_tokens
        count += 1
    return count
```

**This greedy algorithm is what production RAG systems use.** In practice, you also want to re-rank chunks by relevance score before applying the budget, so the highest-quality chunks are added first — not just the first ones returned.

</details>

---
## What the Transformer Is Actually Doing

You don't need to implement a transformer to use LLMs, but knowing the sketch helps.

**At its core:** attention is a weighted average of all tokens in context. Each token "looks at" all other tokens and decides how much to attend to each one. This happens in parallel across all tokens, which is why transformers train so much faster than RNNs.

**Why context matters:** Every token can see every other token in the context window (in the self-attention layers). This means that the word "it" at position 800 can attend back to "the model" at position 12 and understand the reference. This is what makes transformers better at long-range dependencies than earlier sequence models.

**What an embedding is:** When you call `model.encode("some sentence")` with `all-MiniLM-L6-v2`, you're getting the output of this attention process for the `[CLS]` (classification) token — a compressed representation of the full sentence's meaning after attending across all tokens. That's why it's called an *embedding*: the model has embedded the meaning of the sentence into a fixed-size vector.

**The key numbers:**
- `all-MiniLM-L6-v2`: 6 transformer layers, 384-dim embedding, max 512 tokens
- `text-embedding-3-large` (OpenAI): 3072-dim embedding, ~8191 tokens
- `gemini-2.0-flash`: 1M token context, but the embedding dimension is internal

---
## Looking Forward

You now understand the mechanics of how LLMs receive and process text:

- **Tokens** are the real unit of measurement — not characters, not words
- **Context windows** are the hard limit on what you can send
- **API calls** are just messages → text back, with token counts attached
- **Temperature** controls how deterministic the model is
- **Token budgets** are the arithmetic you do before every RAG call

**Next notebook (03):** Prompting as engineering — how to structure your messages to get reliable, structured output. This is what the RAG system prompt is doing, and it's more nuanced than it looks.

---
*Built for [ML Edge](https://mle-edge.dev) — self-directed ML curriculum*